# Détection d'Anomalies dans les Logs Réseau

**Jeu de données :** KDD Cup 1999 (`sklearn.datasets.fetch_kddcup99`)  
**Modèles :** Isolation Forest · Local Outlier Factor · Autoencoder PyTorch  
**Tâche :** Détection non supervisée d'anomalies dans le trafic réseau  

---

Ce notebook compare deux paradigmes de détection d'anomalies appliqués au trafic réseau : l'apprentissage automatique classique (Isolation Forest, Local Outlier Factor) et l'apprentissage profond (Autoencoder). Les modèles sont entraînés exclusivement sur du trafic normal, puis évalués sur l'ensemble complet — approche dite *semi-supervisée*, qui simule un scénario réaliste où les signatures d'attaques sont inconnues au moment de l'entraînement.

Cette approche diffère fondamentalement de la classification supervisée utilisée dans le projet *Adversarial Attacks on IDS* (NSL-KDD). Ici, aucune étiquette n'est utilisée pendant l'entraînement : les modèles apprennent uniquement la structure du trafic normal et signalent tout écart comme anomalie potentielle.

**Structure du notebook :**
1. Chargement et analyse exploratoire du jeu de données
2. Prétraitement
3. Isolation Forest et Local Outlier Factor
4. Autoencoder PyTorch
5. Analyse comparative
6. Synthèse et perspectives

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
import importlib.util, subprocess, sys

PAQUETS = {
    'torch'      : 'torch',
    'sklearn'    : 'scikit-learn',
    'pandas'     : 'pandas',
    'numpy'      : 'numpy',
    'matplotlib' : 'matplotlib',
    'seaborn'    : 'seaborn',
}

manquants = [
    pip_nom for import_nom, pip_nom in PAQUETS.items()
    if importlib.util.find_spec(import_nom) is None
]

if manquants:
    print(f'Installation de : {manquants}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + manquants)
    print('Installation terminée.')
else:
    print('Tous les paquets sont déjà disponibles.')

---
## 1. Chargement et Analyse Exploratoire du Jeu de Données

Le jeu de données KDD Cup 1999 est le benchmark historique de référence pour la détection d'intrusions réseau. Il a été produit à partir de simulations de trafic réseau militaire par le MIT Lincoln Laboratory, dans le cadre du programme DARPA 1998.

Nous utilisons le sous-ensemble **HTTP** (`subset='http'`), qui contient exclusivement du trafic HTTP — réduisant le bruit lié à la diversité des protocoles et permettant une analyse plus ciblée des anomalies applicatives.

Le jeu de données est accessible nativement via `sklearn.datasets.fetch_kddcup99`, sans dépendance externe.

**Catégories d'attaques présentes :**
- **DoS** (*Denial of Service*) : saturation des ressources réseau
- **R2L** (*Remote to Local*) : accès non autorisé depuis une machine distante
- **U2R** (*User to Root*) : élévation de privilèges locale
- **Probe** : reconnaissance et scan de réseau

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.datasets import fetch_kddcup99

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

os.makedirs('../data', exist_ok=True)
print('Bibliothèques chargées avec succès.')

In [ ]:
# Chargement via scikit-learn (KDD Cup 99 HTTP)
# Fallback automatique sur NSL-KDD (GitHub) si figshare est inaccessible

SOURCE = None

try:
    kdd = fetch_kddcup99(subset='http', as_frame=True, percent10=True)
    df  = kdd.frame.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(lambda x: x.decode() if isinstance(x, bytes) else x)
    SOURCE = 'KDD Cup 1999 HTTP (sklearn)'
    print(f'Source : {SOURCE}')

except Exception as e:
    print(f'KDD99 inaccessible ({e}) — bascule sur NSL-KDD (GitHub).')
    NOMS_COLONNES = [
        'duration', 'protocol_type', 'service', 'flag',
        'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
        'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
        'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
        'num_shells', 'num_access_files', 'num_outbound_cmds',
        'is_host_login', 'is_guest_login', 'count', 'srv_count',
        'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
        'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
        'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
        'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
        'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
        'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
        'dst_host_srv_rerror_rate', 'labels', 'difficulty_level'
    ]
    URL_NSL = 'https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt'
    df = pd.read_csv(URL_NSL, names=NOMS_COLONNES).drop(columns=['difficulty_level'])
    SOURCE = 'NSL-KDD (GitHub, fallback)'
    print(f'Source : {SOURCE}')

print(f'Dimensions : {df.shape[0]:,} flux x {df.shape[1]} colonnes')
print(f'Classes    : {df["labels"].unique()}')

### 1.1 Aperçu du Jeu de Données

Le jeu de données KDD99 HTTP contient 41 features décrivant les connexions réseau. Elles se répartissent en trois catégories :

- **Features de base** : caractéristiques de la connexion TCP/IP (durée, protocole, octets source/destination, flags)
- **Features de contenu** : informations extraites du payload (tentatives de login, commandes root, fichiers accédés)
- **Features de trafic** : statistiques sur les connexions récentes vers le même hôte ou service (taux d'erreurs, taux de services différents)

In [ ]:
df.head(5)

In [ ]:
features_numeriques = df.select_dtypes(include=np.number).columns.tolist()
features_cat        = df.select_dtypes(include='object').columns.tolist()

print(f'Features numériques  : {len(features_numeriques)}')
print(f'Features catégorielles : {features_cat}')
print(f'Valeurs manquantes   : {df.isnull().sum().sum()}')
print(f'Valeurs infinies     : {np.isinf(df[features_numeriques]).sum().sum()}')

### 1.2 Distribution des Classes

In [ ]:
# Binarisation pour la visualisation : normal vs attaque
df['label_bin'] = (df['labels'] != 'normal.').astype(int)
distribution_bin = df['label_bin'].value_counts().rename({0: 'Normal', 1: 'Attaque'})
distribution_raw = df['labels'].value_counts()

print('Distribution binaire (Normal / Attaque) :')
print(distribution_bin.to_frame('effectif').assign(
    pourcentage=lambda d: (d['effectif'] / d['effectif'].sum() * 100).round(2)
))

print('\nDistribution par type d\'attaque :')
print(distribution_raw.to_frame('effectif').assign(
    pourcentage=lambda d: (d['effectif'] / d['effectif'].sum() * 100).round(2)
))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

distribution_bin.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Distribution Binaire (Normal / Attaque)')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Effectif')
axes[0].tick_params(axis='x', rotation=0)

distribution_raw.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Distribution par Type d\'Attaque')
axes[1].set_xlabel('Type')
axes[1].set_ylabel('Effectif')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../data/distribution_classes.png', bbox_inches='tight')
plt.show()

### 1.3 Analyse des Features

In [ ]:
df[features_numeriques[:10]].describe().round(3)

In [ ]:
# Features les plus discriminantes (variance non nulle)
variance = df[features_numeriques].var()
features_var = variance[variance > 0].index.tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(
    df[features_var[:12]].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    linewidths=0.5, annot_kws={'size': 7}
)
plt.title('Matrice de Corrélation — 12 Features à Variance Non Nulle')
plt.tight_layout()
plt.savefig('../data/correlation.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribution des features clés par classe
features_plot = [f for f in ['duration', 'src_bytes', 'dst_bytes', 'count']
                 if f in df.columns and df[f].var() > 0]

fig, axes = plt.subplots(1, len(features_plot), figsize=(14, 4))
if len(features_plot) == 1:
    axes = [axes]

for ax, feat in zip(axes, features_plot):
    for label_val, couleur, nom in zip([0, 1], ['steelblue', 'tomato'], ['Normal', 'Attaque']):
        valeurs = df[df['label_bin'] == label_val][feat]
        valeurs = valeurs.clip(valeurs.quantile(0.01), valeurs.quantile(0.99))
        ax.hist(valeurs, bins=50, alpha=0.6, label=nom, color=couleur, density=True)
    ax.set_title(feat)
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Densité')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../data/distribution_features.png', bbox_inches='tight')
plt.show()

---
## 2. Prétraitement

Le prétraitement suit une stratégie semi-supervisée :

1. **Suppression des features à variance nulle** — non informatives pour la détection
2. **Encodage one-hot** des features catégorielles (`protocol_type`, `service`, `flag`)
3. **Binarisation des étiquettes** : `normal. → 0`, toute attaque `→ 1`
4. **Normalisation** : StandardScaler ajusté sur le trafic normal uniquement
5. **Découpage** : entraînement sur trafic normal (85%), évaluation sur l'ensemble complet

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch

# Suppression features à variance nulle
features_utiles = [f for f in features_numeriques if df[f].var() > 0]

# Encodage one-hot des features catégorielles (hors label)
features_cat_model = [c for c in features_cat if c not in ('labels',)]
df_enc = pd.get_dummies(df[features_utiles + features_cat_model + ['label_bin']],
                        columns=features_cat_model)

COLONNE_LABEL = 'label_bin'
feature_cols  = [c for c in df_enc.columns if c != COLONNE_LABEL]

X = df_enc[feature_cols].values.astype(np.float32)
y = df_enc[COLONNE_LABEL].values

print(f'Dimension du vecteur de features : {X.shape[1]}')
print(f'Trafic normal  : {(y == 0).sum():,}')
print(f'Trafic attaque : {(y == 1).sum():,}')

In [ ]:
# Stratégie semi-supervisée
X_normal = X[y == 0]
X_train, X_val = train_test_split(X_normal, test_size=0.15, random_state=42)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_all_s   = scaler.transform(X)

print(f'Entraînement (normal)  : {X_train_s.shape[0]:,} échantillons')
print(f'Validation  (normal)   : {X_val_s.shape[0]:,} échantillons')
print(f'Évaluation  (complet)  : {X_all_s.shape[0]:,} échantillons')

---
## 3. Modèles de Machine Learning Classique

### 3.1 Isolation Forest

L'Isolation Forest (Liu et al., 2008) isole les anomalies via des arbres de décision aléatoires. Les points anormaux, naturellement rares et distincts, requièrent moins de coupures pour être isolés — leur chemin moyen dans les arbres est plus court. Le score d'anomalie est inversement proportionnel à cette longueur de chemin.

### 3.2 Local Outlier Factor

Le LOF (Breunig et al., 2000) mesure la densité locale d'un point relativement à ses voisins. Un point dont la densité est nettement inférieure à celle de ses voisins reçoit un score LOF élevé et est considéré comme une anomalie.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve,
    average_precision_score, precision_score,
    recall_score, f1_score
)

CONTAMINATION = float((y == 1).mean())
print(f'Taux de contamination : {CONTAMINATION:.4f}')

In [ ]:
print('Entraînement Isolation Forest...')
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=CONTAMINATION,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train_s)

y_pred_if = (iso_forest.predict(X_all_s) == -1).astype(int)
scores_if  = -iso_forest.score_samples(X_all_s)

print('\nIsolation Forest — Rapport de classification :')
print(classification_report(y, y_pred_if, target_names=['Normal', 'Attaque']))
print(f'AUC-ROC           : {roc_auc_score(y, scores_if):.4f}')
print(f'Average Precision : {average_precision_score(y, scores_if):.4f}')

In [ ]:
print('Entraînement Local Outlier Factor...')
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=CONTAMINATION,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_train_s)

y_pred_lof = (lof.predict(X_all_s) == -1).astype(int)
scores_lof  = -lof.score_samples(X_all_s)

print('\nLocal Outlier Factor — Rapport de classification :')
print(classification_report(y, y_pred_lof, target_names=['Normal', 'Attaque']))
print(f'AUC-ROC           : {roc_auc_score(y, scores_lof):.4f}')
print(f'Average Precision : {average_precision_score(y, scores_lof):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_pred, titre in zip(
    axes,
    [y_pred_if, y_pred_lof],
    ['Isolation Forest', 'Local Outlier Factor']
):
    mat = confusion_matrix(y, y_pred)
    sns.heatmap(mat, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Attaque'],
                yticklabels=['Normal', 'Attaque'])
    ax.set_title(f'Matrice de Confusion — {titre}')
    ax.set_ylabel('Étiquette réelle')
    ax.set_xlabel('Étiquette prédite')
plt.tight_layout()
plt.savefig('../data/confusion_ml.png', bbox_inches='tight')
plt.show()

---
## 4. Autoencoder PyTorch

Un autoencoder est un réseau de neurones entraîné à reconstruire ses entrées via un goulot d'étranglement (*bottleneck*). Entraîné exclusivement sur du trafic normal, il apprend à reconstruire fidèlement ce type de trafic. Face à du trafic anormal, l'erreur de reconstruction (MSE) est élevée — ce signal constitue le score d'anomalie.

Le seuil de détection est déterminé empiriquement à partir du percentile 95 des erreurs de reconstruction sur l'ensemble de validation (trafic normal uniquement).

**Architecture :**

| Composant | Couches | Activation |
|-----------|---------|------------|
| Encodeur | `dim_entrée → 64 → 32 → 16` | ReLU |
| Décodeur | `16 → 32 → 64 → dim_entrée` | ReLU / Identité |

**Perte :** Mean Squared Error (MSE)

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

GRAINE = 42
torch.manual_seed(GRAINE)
np.random.seed(GRAINE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositif : {DEVICE}')

DIM_ENTREE = X_train_s.shape[1]

class Autoencoder(nn.Module):
    def __init__(self, dim_entree: int, dim_latent: int = 16):
        super().__init__()
        self.encodeur = nn.Sequential(
            nn.Linear(dim_entree, 64), nn.ReLU(),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, dim_latent), nn.ReLU(),
        )
        self.decodeur = nn.Sequential(
            nn.Linear(dim_latent, 32), nn.ReLU(),
            nn.Linear(32, 64),         nn.ReLU(),
            nn.Linear(64, dim_entree),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decodeur(self.encodeur(x))

    def erreur_reconstruction(self, x: torch.Tensor) -> np.ndarray:
        self.eval()
        with torch.no_grad():
            return ((x - self.forward(x)) ** 2).mean(dim=1).cpu().numpy()


autoencoder = Autoencoder(DIM_ENTREE).to(DEVICE)
nb_params = sum(p.numel() for p in autoencoder.parameters())
print(autoencoder)
print(f'\nNombre de paramètres entraînables : {nb_params:,}')

In [ ]:
NB_EPOQUES   = 60
TAILLE_BATCH = 512
PATIENCE     = 7

X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_s,   dtype=torch.float32)
X_all_t   = torch.tensor(X_all_s,   dtype=torch.float32)

loader_train = DataLoader(TensorDataset(X_train_t), batch_size=TAILLE_BATCH, shuffle=True)
loader_val   = DataLoader(TensorDataset(X_val_t),   batch_size=TAILLE_BATCH, shuffle=False)

critere    = nn.MSELoss()
optimiseur = optim.Adam(autoencoder.parameters(), lr=1e-3)

historique = {'train': [], 'val': []}
meilleure_val, compteur, meilleur_etat = float('inf'), 0, None

for epoque in range(1, NB_EPOQUES + 1):
    autoencoder.train()
    perte_train = 0.0
    for (xb,) in loader_train:
        xb = xb.to(DEVICE)
        optimiseur.zero_grad()
        perte = critere(autoencoder(xb), xb)
        perte.backward()
        optimiseur.step()
        perte_train += perte.item() * len(xb)

    autoencoder.eval()
    perte_val = 0.0
    with torch.no_grad():
        for (xb,) in loader_val:
            xb = xb.to(DEVICE)
            perte_val += critere(autoencoder(xb), xb).item() * len(xb)

    perte_train /= len(X_train_t)
    perte_val   /= len(X_val_t)
    historique['train'].append(perte_train)
    historique['val'].append(perte_val)

    if epoque % 10 == 0:
        print(f'Époque {epoque:3d}/{NB_EPOQUES}  train={perte_train:.6f}  val={perte_val:.6f}')

    if perte_val < meilleure_val:
        meilleure_val, compteur = perte_val, 0
        meilleur_etat = {k: v.clone() for k, v in autoencoder.state_dict().items()}
    else:
        compteur += 1
        if compteur >= PATIENCE:
            print(f'Arrêt anticipé à l\'époque {epoque}.')
            break

autoencoder.load_state_dict(meilleur_etat)
print('Meilleur modèle restauré.')

In [ ]:
epoques = range(1, len(historique['train']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epoques, historique['train'], label='Entraînement', color='steelblue')
plt.plot(epoques, historique['val'],   label='Validation',   color='tomato', linestyle='--')
plt.title('Évolution de la Perte MSE — Autoencoder')
plt.xlabel('Époque')
plt.ylabel('Perte MSE')
plt.legend()
plt.tight_layout()
plt.savefig('../data/courbes_autoencoder.png', bbox_inches='tight')
plt.show()

In [ ]:
# Calcul du seuil sur la validation (trafic normal uniquement)
erreurs_val = autoencoder.erreur_reconstruction(X_val_t.to(DEVICE))
erreurs_all = autoencoder.erreur_reconstruction(X_all_t.to(DEVICE))

SEUIL = np.percentile(erreurs_val, 95)
print(f'Seuil de détection (percentile 95) : {SEUIL:.6f}')

y_pred_ae = (erreurs_all > SEUIL).astype(int)

print('\nAutoencoder — Rapport de classification :')
print(classification_report(y, y_pred_ae, target_names=['Normal', 'Attaque']))
print(f'AUC-ROC           : {roc_auc_score(y, erreurs_all):.4f}')
print(f'Average Precision : {average_precision_score(y, erreurs_all):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label_val, couleur, nom in zip([0, 1], ['steelblue', 'tomato'], ['Normal', 'Attaque']):
    axes[0].hist(erreurs_all[y == label_val], bins=100, alpha=0.6,
                 label=nom, color=couleur, density=True)
axes[0].axvline(SEUIL, color='black', linestyle='--', label=f'Seuil ({SEUIL:.4f})')
axes[0].set_title('Distribution des Erreurs de Reconstruction')
axes[0].set_xlabel('Erreur MSE')
axes[0].set_ylabel('Densité')
axes[0].legend()

mat_ae = confusion_matrix(y, y_pred_ae)
sns.heatmap(mat_ae, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal', 'Attaque'],
            yticklabels=['Normal', 'Attaque'])
axes[1].set_title('Matrice de Confusion — Autoencoder')
axes[1].set_ylabel('Étiquette réelle')
axes[1].set_xlabel('Étiquette prédite')

plt.tight_layout()
plt.savefig('../data/autoencoder_viz.png', bbox_inches='tight')
plt.show()

---
## 5. Analyse Comparative

Nous comparons les trois modèles selon cinq métriques. Dans un contexte de sécurité réseau, le **rappel** est la métrique prioritaire : un faux négatif (attaque non détectée) est bien plus coûteux qu'un faux positif (alerte injustifiée). L'**AUC-ROC** mesure la capacité discriminante globale indépendamment du seuil.

In [ ]:
def calculer_metriques(y_true, y_pred, scores):
    return {
        'Précision'        : precision_score(y_true, y_pred, zero_division=0),
        'Rappel'           : recall_score(y_true, y_pred, zero_division=0),
        'F1-score'         : f1_score(y_true, y_pred, zero_division=0),
        'AUC-ROC'          : roc_auc_score(y_true, scores),
        'Average Precision': average_precision_score(y_true, scores),
    }

resultats = pd.DataFrame({
    'Isolation Forest'    : calculer_metriques(y, y_pred_if,  scores_if),
    'Local Outlier Factor': calculer_metriques(y, y_pred_lof, scores_lof),
    'Autoencoder'         : calculer_metriques(y, y_pred_ae,  erreurs_all),
}).T.round(4)

print('=== Tableau Comparatif ===')
resultats

In [ ]:
metriques_plot = ['Précision', 'Rappel', 'F1-score', 'AUC-ROC']
x = np.arange(len(metriques_plot))
width = 0.25
couleurs = ['steelblue', 'seagreen', 'tomato']

fig, ax = plt.subplots(figsize=(10, 5))
for i, (modele, couleur) in enumerate(zip(resultats.index, couleurs)):
    valeurs = [resultats.loc[modele, m] for m in metriques_plot]
    barres = ax.bar(x + i * width, valeurs, width, label=modele, color=couleur)
    for b in barres:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f'{b.get_height():.3f}', ha='center', fontsize=7)

ax.set_xticks(x + width)
ax.set_xticklabels(metriques_plot)
ax.set_ylim(0, 1.15)
ax.set_title('Comparaison des Modèles de Détection d\'Anomalies')
ax.set_ylabel('Score')
ax.legend()
plt.tight_layout()
plt.savefig('../data/comparaison_modeles.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for scores, nom, couleur in zip(
    [scores_if, scores_lof, erreurs_all],
    ['Isolation Forest', 'Local Outlier Factor', 'Autoencoder'],
    ['steelblue', 'seagreen', 'tomato']
):
    prec_c, rap_c, _ = precision_recall_curve(y, scores)
    ap = average_precision_score(y, scores)
    ax.plot(rap_c, prec_c, label=f'{nom} (AP={ap:.3f})', color=couleur, linewidth=2)

ax.set_xlabel('Rappel')
ax.set_ylabel('Précision')
ax.set_title('Courbes Précision-Rappel')
ax.legend()
plt.tight_layout()
plt.savefig('../data/courbes_pr.png', bbox_inches='tight')
plt.show()

---
## 6. Synthèse et Perspectives

### Bilan

Cette étude compare trois approches non supervisées de détection d'anomalies réseau sur le jeu de données KDD Cup 99 (sous-ensemble HTTP), dans un cadre semi-supervisé où les modèles ne voient aucune attaque à l'entraînement.

**Isolation Forest** offre le meilleur compromis entre rapidité d'entraînement et performance. Son score d'anomalie continu est directement exploitable pour du triage d'alertes.

**Local Outlier Factor** est plus sensible aux anomalies contextuelles locales mais reste coûteux en inférence sur de grands volumes de trafic.

**L'Autoencoder** apprend une représentation compacte du trafic normal et détecte les écarts via l'erreur de reconstruction. Il excelle face à des anomalies complexes non linéaires. Le choix du seuil de détection (ici percentile 95) est un hyperparamètre critique qui pilote le compromis précision/rappel.

### Comparaison avec l'approche supervisée

Par rapport à la classification supervisée (projet *Adversarial Attacks on IDS*, NSL-KDD), l'approche non supervisée présente un rappel généralement plus faible mais s'applique sans étiquetage des attaques — ce qui est réaliste en production où de nouvelles attaques apparaissent continuellement.

### Perspectives

- Extension à d'autres sous-ensembles KDD99 (smtp, ftp)
- Variantes d'autoencoders : Variational Autoencoder (VAE), LSTM Autoencoder pour les séries temporelles
- Détection en temps réel sur flux de paquets (*streaming*)
- Interprétabilité via SHAP values sur l'Isolation Forest